In [1]:
from typing import Iterable, Iterator, Callable
from itertools import chain as iterchain, combinations as itercomb
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [2]:
from board import DIGITS, Digits, Loc, Cell, Board

# Topology

The board is classicaly 9 rows, 9 columns, and 9 boxes over them. These are major units.

Intersection of a row and a column gives individual cell. A cell is subdivided further into 9 segments for digits, but only for visualizing purposes.

Intersection of a box with a row or column gives sector.

Generalized, all the localities can be addressed by a set of `(box, col, row)`

Visibility of 2 localities means they share some unit (or two). It determines location of cell peers and possibly conflicting drafts.


In [3]:
from topology import Zone, peers, peerz, allpeers, visibility, allvisible

In [4]:
# construction
assert Zone.B(1) == Zone(1, ..., ...)
assert Zone.R(2) == Zone(..., 2, ...)
assert Zone.C(3) == Zone(..., ..., 3)
assert Zone.L(Loc(2, 3)) == Zone(1, 2, 3)  # fulfiled box

assert Zone(1, 2, 3).is_cell
assert Zone(1, ..., ...).is_unit
assert Zone(1, 2, ...).is_sector

# intersection
assert Zone.R(2) & Zone.C(3) == Zone(1, 2, 3)
assert Zone.B(1) & Zone.R(2) == Zone(1, 2, ...)
assert Zone.B(1) & Zone.C(3) == Zone(1, ..., 3)

# containering
assert set(iter(Zone(1, ..., ...))) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert set(iter(Zone(1, 2, ...))) == {Loc(2, 1), Loc(2, 2), Loc(2, 3)}
assert Loc(2, 3) in Zone.B(1)
assert Loc(2, 3) in Zone.R(2)
assert Loc(2, 3) in Zone.C(3)

In [5]:
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone.B(1)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(1, 3))) == {Zone.R(1), Zone.B(1)}
assert visibility(Zone(1, 2, ...), Zone.L(Loc(2, 9))) == {Zone.R(2)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == set()

assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == {Zone(3, 1, 9), Zone(7, 8, 2)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone(1, ..., ...)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(3, 2))) == {Zone(1, ..., ...), Zone(..., ..., 2)}

In [6]:
print(*map(str, Zone.Units()))

b1…… b2…… b3…… b4…… b5…… b6…… b7…… b8…… b9…… …r1… …r2… …r3… …r4… …r5… …r6… …r7… …r8… …r9… ……c1 ……c2 ……c3 ……c4 ……c5 ……c6 ……c7 ……c8 ……c9


In [7]:
assert peers(Loc(1, 2), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert peerz(Zone(1, 2, ...), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}

# Targeting

Generic draft target:

- location: cell, unit, sector
- digits: one or more digits


In [8]:
from analysis import Node, cellmatching, cellspoiling

In [9]:
assert Node.C(Cell(Loc(1, 2), Digits({1, 2, 3}))) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), Digits({1, 2, 3})) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), 5) == Node(Zone.L(Loc(1, 2)), Digits({5}))

assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_cellular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_singular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_casual
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_singular
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_casual
assert not Node(Zone(1, 2, ...), Digits({1})).is_cellular
assert not Node(Zone(1, 2, ...), Digits({1})).is_casual

# Resolving

searching for patterns -> generating resolutions -> applying resolutions


In [10]:
from utils import filt_finals, filt_having, filt_havesome, draftborhood, draftboard, flat_cells, count_finals, count_digits
from solving import Resolving, Resolver, Resolution, solver
from searching import search_breadth
from analysis import Link, HLink, SLink, Chain

In [11]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    _, _, drafted = result.validate()
    assert drafted
    async for _, _, result in solver(initial, *resolvers):
        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break
    return result

In [12]:
async def solve_logging(initial: Board, /, *resolvers: Resolver, filtout: set[str] | None = None):
    result = initial
    iterations = 0
    _, _, drafted = result.validate()
    assert drafted
    async for resolver, resolution, result in solver(initial, *resolvers):
        iterations += 1
        if filtout and resolver.__name__ in filtout:
            continue
        print(f"{iterations:03d} {resolver.__name__}", end=": ")
        if resolution.castaways:
            print("-= {", " ".join(map(str, resolution.castaways)), "}", end=" ")
        if resolution.finals:
            print(":= {", " ".join(map(str, resolution.finals)), "}", end=" ")
        if resolution.highlights:
            if "zone" in resolution.highlights:
                print("@", resolution.highlights["zone"], end=" ")
            print("#", end=" ")
            if "anchors" in resolution.highlights:
                print(" ".join(map(str, resolution.highlights["anchors"])), end=" ")
            if "chain" in resolution.highlights:
                print(resolution.highlights["chain"], end=" ")
        print()

        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break

    complete, valid, drafted = result.validate()
    stuck = not complete and not drafted
    print("========")
    print(f"{iterations=} {complete=} {valid=} {stuck=}")
    return result


## Random choice


In [13]:
def randomchoice(board: Board) -> Resolving:
    """Last resort move for non-uniq puzzles"""
    drafts = list(draftboard(board))
    assert len(drafts)
    drafts.sort(key=lambda c: len(c))
    counts = count_digits(drafts)
    leastcell = drafts[0]
    digits = list(leastcell.digits)
    digits.sort(key=lambda d: counts[d])
    leastdig = digits[0]
    yield Resolution(castaways={Cell(leastcell.loc, Digits({leastdig}))})

## Singles


In [14]:
def open_singles(board: Board) -> Resolving:
    """Open singles (finals): removing drafts conflicting with finals"""

    for fincell in filter(filt_finals, iter(board)):
        anchor = Node.at(fincell, fincell.final)
        spoilers = set(cellmatching(anchor, draftborhood(board, allpeers(anchor.zone))))
        if len(spoilers):
            yield Resolution(
                castaways=spoilers,
                highlights={
                    "anchors": {anchor},
                },
            )

In [15]:
def hidden_singles(board: Board) -> Resolving:
    """Hidden finals: removing all other drafts from the cell of a single"""

    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))
        counts = count_digits(drafts)
        for dig in (d for d in DIGITS if counts[d] == 1):
            stretch = Node.at(unit, dig)
            [lonesome] = cellmatching(stretch, drafts)
            yield Resolution(
                finals={lonesome},
                highlights={
                    "zone": unit,
                    "anchors": {Node.C(lonesome)},
                    "empties": {stretch},
                },
            )

## Multiples


In [16]:
def open_mults(board: Board, mult: int) -> Resolving:
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        counts = count_digits(drafts)
        families = {
            Digits(combo): set(filter(filt_havesome(set(combo)), drafts))
            for combo in itercomb(counts.keys(), mult)  # hell lot!
        }
        if not len(families):
            continue
        for combo, family in families.items():
            naked = set(filter(lambda c: c.digits <= combo, family))
            if len(naked) != mult:
                continue

            stretch = Node(unit, combo)
            spoilers = set(cellmatching(stretch, drafts)) - naked
            if spoilers:
                yield Resolution(
                    castaways=spoilers,
                    highlights={
                        "zone": unit,
                        "anchors": {Node.C(c) for c in naked},
                    },
                )

In [17]:
def open_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return open_mults(board, mult)

    resolver.__name__ = f"open_mults[{mult}]"
    return resolver

In [18]:
def hidden_mults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        counts = count_digits(drafts)
        families = {
            Digits(combo): set(filter(filt_havesome(set(combo)), drafts))
            for combo in itercomb(counts.keys(), mult)  # hell lot!
        }
        if not len(families):
            continue
        for combo, family in families.items():
            if len(family) != mult:
                continue
            stretch = Node(unit, combo)
            spoilers = set(cellspoiling(stretch, family))
            if spoilers:
                yield Resolution(
                    castaways=spoilers,
                    highlights={
                        "zone": unit,
                        "anchors": set(Node.C(c) for c in cellmatching(stretch, family)),
                        "empties": {stretch},
                    },
                )


def hidden_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return hidden_mults(board, mult)

    resolver.__name__ = f"hidden_mults[{mult}]"
    return resolver

## Triplets


In [19]:
def search_groups(board: Board) -> Iterator[Node]:
    """Search for all groups at sectors"""
    for unit in Zone.Allbox():
        for sect in Zone.sectors(unit):
            counts = count_digits(draftborhood(board, sect))
            for dig, cnt in counts.items():
                if cnt == 1:
                    continue
                yield Node.at(sect, dig)

In [20]:
def triplets(board: Board) -> Resolving:
    for group in search_groups(board):
        units = tuple(Zone.around(group.zone))  # just 2 of them
        crossborhoods = {unit: set(cellmatching(group, draftborhood(board, peerz(group.zone, unit)))) for unit in units}
        spacious = tuple(u for u in units if len(crossborhoods[u]) == 0)
        spoilerhoods = tuple(u for u in units if len(crossborhoods[u]) > 0)
        if len(spacious) != 1 or len(spoilerhoods) != 1:
            continue
        spacious = spacious[0]
        spoiled = spoilerhoods[0]
        yield Resolution(
            castaways=crossborhoods[spoiled],
            highlights={
                "zone": spoiled,
                "empties": {Node(spacious, group.digits)},
                "anchors": {group},
            },
        )

## Links/Chains


### Strong/Hard links

Criteria:

- only 2 drafts of same digit in a unit
- only 2 drafts in a cell


In [21]:
def search_hard1(board: Board) -> Iterable[HLink]:
    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(n1, dig), Node.at(n2, dig)))

In [22]:
def search_hard2(board: Board):
    for cell in filter(lambda c: len(c) == 2, draftboard(board)):
        d1, d2 = cell.digits
        yield HLink((Node.at(cell, d1), Node.at(cell, d2)))


### Weak/Soft links

Criteria:

- any 2 drafts of same digit in a unit
- any 2 draft in a cell


In [ ]:
def connect_soft12(n1: Node, n2: Node):
    assert n1 != n2
    assert all(n.is_casual for n in (n1, n2))
    if n1.dig == n2.dig and n1.zone != n2.zone and visibility(n1.zone, n2.zone):
        # bilocation
        return SLink((n1, n2))
    if n1.dig != n2.dig and n1.zone == n2.zone:
        return SLink((n1, n2))
    return None

### Chains

Alternating inference chains, with simple bilocation links

X-Wing, X-Cycle, Nice Loop, etc


In [ ]:
def search_softable1(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    """Sca for all nodes soft-visible from both e1 and e2"""
    assert all(n.is_casual for n in (n1, n2))

    anchors = n1.digits | n2.digits  # max=2
    anclocs = {n1.zone.loc(), n2.zone.loc}
    for vizone in allvisible(n1.zone, n2.zone):  # max=2
        for cell in filter(lambda c: c.digits & anchors and c.loc not in anclocs, draftborhood(board, vizone)):  # max=18
            for dig in anchors:  # max=36
                node = Node.at(cell, dig)
                if connect_soft12(node, n1) and connect_soft12(node, n2):
                    yield node

In [25]:
def search_softable2(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    assert all(n.is_casual for n in (n1, n2))
    if n1.zone != n2.zone:
        return
    cell = board.get(n1.zone.loc())
    anchors = n1.digits | n2.digits
    for d in filter(lambda d: d not in anchors, cell.digits):
        yield Node.at(cell, d)


In [26]:
def search_softable12(board: Board, n1: Node, n2: Node) -> set[Node]:
    return set(search_softable1(board, n1, n2)) | set(search_softable2(board, n1, n2))

#### open

Pattern: a unclosed alternating hard-ended chain with matching edges

Rule: invalidate all visible from both edges


In [ ]:
def match_rope12(chain: Chain):
    assert all(n.is_casual for n in chain.edges)
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and chain.is_hardend and e1.dig == e2.dig


def resolve_rope12(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    assert all(n.is_casual for n in chain.edges)
    e1, e2 = chain.edges
    anchors = chain.anchors()
    spoilers = set(search_softable12(board, e1, e2)) - anchors
    if len(spoilers):
        yield Resolution(
            castaways={n.cell() for n in spoilers},
            highlights={
                "anchors": {e1, e2},
                "chain": chain,
            },
        )

#### loop

Pattern: a closed alternating chain (odd number of links)

Rule: invalidate all visible from both edges of each soft link


In [28]:
def match_loop12(chain: Chain):
    assert all(n.is_casual for n in chain.edges)
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop12(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges of each soft link"""
    assert all(n.is_casual for n in chain.edges)
    anchors = chain.anchors()  # to exclude linking back to chain
    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        e1, e2 = link
        spoilers = set(search_softable12(board, e1, e2)) - anchors
        if len(spoilers):
            yield Resolution(
                castaways={n.cell() for n in spoilers},
                highlights={
                    "anchors": {e1, e2},
                    "chain": chain,
                },
            )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [29]:
def expand_chain12(current: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 1 and chain.is_hardend:
            closing = connect_soft12(e2, e1)
            if closing:
                yield Chain.extend(chain, closing)

    def extend(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if conn := connect_soft12(e2, x1):
            yield Chain.extend(chain, conn, link)
        if conn := connect_soft12(e2, x2):
            yield Chain.extend(chain, conn, link.reversed())
        # if conn := connect_soft1(x2, e1):
        #     yield Chain.extendhead(chain, link, conn)
        # if conn := connect_soft1(x1, e1):
        #     yield Chain.extendhead(chain, link.reversed(), conn)

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        for extended in extend(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [ ]:
def search_chains12(current: Board, links: Iterable[HLink], matching: Callable[[Chain], bool], max_length: int = 8):
    counts = count_finals(current)

    def rate(link: Link):
        return min(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain((l,)) for l in links]

    def expanding(chain: Chain):
        yield from expand_chain12(chain, links)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

In [133]:
def match_useful(chain: Chain):
    return match_loop12(chain) or match_rope12(chain)


In [134]:
def chains12(current: Board) -> Resolving:
    """Resolve simple chains"""
    links = set(search_hard1(current)) | set(search_hard2(current))

    for chain in search_chains12(current, links, match_useful, max_length=6):
        if match_loop12(chain):
            res = tuple(resolve_loop12(current, chain))
        elif match_rope12(chain):
            res = tuple(resolve_rope12(current, chain))
        else:
            res = None

        if not res:
            continue  # if didn't work

        yield from res
        break  # on first worked

# A puzzle


In [221]:
from utils import fillempty, picture, parsepic

puzzle = parsepic("""
+--------------+---------------+--------------+
| 8   4   2    | 56  56   3    | 7   1   9    |
| 67  679 3    | 1   789  4    | 568 258 26   |
| 5   679 1    | 27  2789 289  | 68  3   4    |
+--------------+---------------+--------------+
| 69  3   8    | 26  1    269  | 4   7   5    |
| 49  2   45   | 3   59   7    | 1   6   8    |
| 1   56  7    | 456 458  568   | 2   9   3    |
+--------------+---------------+--------------+
| 3   58  6    | 247 2457  25   | 9   458 1    |
| 47  578 45   | 9   3    1    | 568 258 26   |
| 2   1   9    | 8   456   56   | 3   45  7    |
+--------------+---------------+--------------+
""")

puzzle = Board.transform(puzzle, fillempty)

In [161]:
puzzle = await solve_silent(puzzle, open_singles, hidden_singles)

In [ ]:
await solve_logging(
    puzzle,
    open_singles,
    hidden_singles,
    triplets,
    open_mults_(2),
    hidden_mults_(2),
    open_mults_(3),
    hidden_mults_(3),
    open_mults_(4),
    hidden_mults_(4),
    open_mults_(5),
    hidden_mults_(5),
    chains12,
    # randomchoice,
    filtout={"open_singles", "hidden_singles"},
)

# GUI


In [51]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [35]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Bool, Dict
from canvas import SudokuCanvas

In [36]:
def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


# TODO: make it cancellable somehow

In [ ]:
class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Unicode()
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Cell))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for lnk in self.links:
                self._highlight_link(lnk, "blue")

            for node in self.empties:
                self._highlight_node(node, "pink")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for cell in self.targets:
                self._highlight_cell(cell, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in node.zone:
            for dig in node.digits:
                self._canvas.highlight_segment(loc, dig, color=color)

    def _highlight_cell(self, cell: Cell, color: str):
        for dig in cell.digits:
            self._canvas.highlight_segment(cell.loc, dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        n1, n2 = lnk
        if n1.zone.is_cell and n2.zone.is_cell:
            self._canvas.highlight_link(
                n1.loc,
                n1.dig,
                n2.loc,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            n1locs = tuple(iter(n1.zone))
            l1mid = Loc(
                sum(l.r for l in n1locs) // len(n1locs),
                sum(l.c for l in n1locs) // len(n1locs),
            )
            n2locs = tuple(iter(n2.zone))
            l2mid = Loc(
                sum(l.r for l in n2locs) // len(n2locs),
                sum(l.c for l in n2locs) // len(n2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                n1.dig,
                l2mid,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig, lmax, node.dig, style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False


# GUI meta-widget


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [171]:
debug_view = w.Output()
gui = GUI()

In [173]:
display(gui, debug_view)

Output()

In [174]:
gui.puzzle = puzzle
gui.targets = set()
gui.anchors = set()
gui.links = set()
gui.status = "..."

In [ ]:
links = set(search_hard1(puzzle)) | set(search_hard2(puzzle))
links = set(filter(lambda l: l[0].digits & {1, 5, 9} and l[1].digits & {1, 5, 9}, links))
searching = search_chains12(puzzle, links, match_useful)
gui.links = links

In [ ]:
chain = next(searching)
gui.links = set(chain)

In [41]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial

    _, _, drafted = result.validate()
    assert drafted

    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, resolution, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(resolution)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: "
                if resolution.castaways:
                    resolving += f"-= {len(resolution.castaways)}"
                if resolution.finals:
                    resolving += f"== {len(resolution.finals)}"
                gui.resolving = resolving
                render_resolution(resolution)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"

            complete, valid, drafted = result.validate()
            render_status(complete, valid, drafted)
            if complete or not valid or not drafted:
                break
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    complete, valid, drafted = result.validate()
    render_status(complete, valid, drafted)
    gui.resolving = f"#{iteration} ENDED"
    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    with hold_canvas():
        gui.targets = res.castaways if res.castaways else set()

        gui.anchors = res.highlights.get("anchors", set())
        gui.empties = res.highlights.get("empties", set())

        if "chain" in res.highlights:
            gui.links = set(res.highlights["chain"])
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def render_status(complete: bool, valid: bool, drafted: bool):
    if not valid:
        gui.status = "BROKEN"
    elif complete:
        gui.status = "SOLVED"
    elif not drafted:
        gui.status = "STUCK"
    else:
        gui.status = "..."


def clear_resolution():
    gui.reset_highlights()

In [222]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        open_singles,
        hidden_singles,
        triplets,
        open_mults_(2),
        hidden_mults_(2),
        # open_mults_(3),
        # hidden_mults_(3),
        # open_mults_(4),
        # hidden_mults_(4),
        # open_mults_(5),
        # hidden_mults_(5),
        chains12,
        # randomchoice,
        filtout={"open_singles", "hidden_singles"},
    )
)

In [ ]:
task

In [170]:
task.cancel()  # it breaks something

False